# Lab 5 — A small recurrent translator and beam search

COMP-672 · John Glossner

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/glossner/COMP-672-Labs/blob/main/labs/week05_starter.ipynb)

**First: File → Save a copy in Drive.** Sign in to Google to run and save. Rename your copy, for example `COMP672_Lab05_YourName.ipynb`, and work in that copy.


## Install the lab environment

Use Colab’s **default Python CPU runtime**; you do not need to select an older Python runtime. 


In [ ]:
import sys
import subprocess
import importlib.metadata as metadata

if not (3, 11) <= sys.version_info[:2] <= (3, 14):
    raise RuntimeError("This lab supports Python 3.11–3.14. Reopen the latest notebook from the course README if your runtime is newer.")

print("Runtime Python:", sys.version.split()[0])

LAB_PACKAGES = {'numpy': '2.1.3', 'scipy': '1.16.3', 'scikit-learn': '1.6.1', 'matplotlib': '3.10.0'}
# Python 3.14 needs newer binary wheels; retain the established pins on 3.11–3.13.
if sys.version_info[:2] >= (3, 14):
    LAB_PACKAGES.update({'numpy': '2.3.5', 'scikit-learn': '1.7.2', 'matplotlib': '3.10.8'})
LAB_PACKAGES["torch"] = "2.11.0"
def installed_version(name):
    try:
        return metadata.version(name).split("+")[0]
    except metadata.PackageNotFoundError:
        return None

needed = [f"{name}=={version}" for name, version in LAB_PACKAGES.items()
          if name != "torch" and installed_version(name) != version]
if needed:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:", *needed])
if installed_version("torch") != LAB_PACKAGES["torch"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--only-binary=:all:",
                           "torch==2.11.0", "--index-url",
                           "https://download.pytorch.org/whl/cpu"])
print("Package setup complete. Restart the session if Colab requests it.")


In [ ]:
import platform
import importlib

LAB_VERSIONS = {"python": platform.python_version()}
for distribution, module_name in [("numpy", "numpy"), ("scipy", "scipy"),
                                  ("scikit-learn", "sklearn"), ("matplotlib", "matplotlib")]:
    module = importlib.import_module(module_name)
    LAB_VERSIONS[distribution] = module.__version__
import torch
torch.set_num_threads(1)
LAB_VERSIONS["torch"] = torch.__version__
for name, expected in LAB_PACKAGES.items():
    if LAB_VERSIONS[name].split("+")[0] != expected:
        raise RuntimeError(f"Restart the session: {name} is already imported at a different version.")
print(LAB_VERSIONS)
print("CPU environment ready. Synthetic data will be generated by the lab functions.")


# Lab 5 — A small recurrent translator and beam search

Chapters 14, 13

## Implementation tasks

1. Implement normalized source attention and verify the output is a weighted combination of encoder states.

2. Implement a beam with cumulative log scores, EOS handling, a finite generation limit, and deterministic tie breaking.

3. Implement next-target loss for the teacher-forced recurrent decoder. Trace a training step and a free-running step.


Functions to complete: `context_attention`, `beam_search`, `sequence_loss`.

## Required experiments

- Train the small GRU model on the fixed training combinations and evaluate greedy versus width-3 beam search on held-out combinations. Report all individual predictions as well as exact match.

- Run the supplied probability-tree counterexample and verify that beam width 2 finds probability .38 while greedy finds .36. State the no-length-normalization convention.

## Brief analysis in your notebook or code comments

1. Why can teacher-forced loss improve without equivalent free-running performance?

2. Is a beam gain evidence about search or about the learned distribution, and what does the controlled counterexample isolate?

3. What would be needed to extend the invented reordering task to real machine translation?


Answer each question in approximately 2–4 sentences beside the results. This is part of the lab, with no separate report.


## Starter code

Complete the exercise functions below. Keep the supplied checks and add the cases required by the handout.

In [ ]:
"""Week 5 student starter: compositional toy translation, GRU encoder-decoder."""


In [ ]:
import math


In [ ]:
import torch


In [ ]:
from torch import nn


In [ ]:
from torch.nn import functional as F


In [ ]:
torch.set_num_threads(1)


In [ ]:
def context_attention(query,keys):
    """query B,d; keys B,T,d; dot-product attention over source positions."""
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def beam_search(step,bos,eos,width=2,max_tokens=4):
    """step(prefix)->log probabilities. Carry completed hypotheses without expansion.
    Rank raw cumulative log probability; return best completed, else best live prefix.
    """
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
class Translator(nn.Module):
    def __init__(self,vocab=12,d=24):
        super().__init__(); self.emb=nn.Embedding(vocab,d)
        self.encoder=nn.GRU(d,d,batch_first=True); self.cell=nn.GRUCell(2*d,d)
        self.head=nn.Linear(2*d,vocab)
    def encode(self,src):
        keys,h=self.encoder(self.emb(src)); return keys,h[0]
    def decode(self,token,h,keys):
        ctx,_=context_attention(h,keys); h=self.cell(torch.cat([self.emb(token),ctx],-1),h)
        ctx,_=context_attention(h,keys); return self.head(torch.cat([h,ctx],-1)),h
    def teacher_forced(self,src,target):
        keys,h=self.encode(src); outputs=[]
        for t in range(target.size(1)-1):
            logits,h=self.decode(target[:,t],h,keys); outputs.append(logits)
        return torch.stack(outputs,1)


In [ ]:
def sequence_loss(logits,target):
    raise NotImplementedError("Complete this laboratory task.")


In [ ]:
def data():
    # 0 BOS, 1 EOS; source color (2..5), shape (6..9); target shape then color.
    # This invented translation task tests reordering, not natural French proficiency.
    pairs=[(c,s) for c in range(2,6) for s in range(6,10)]
    train=[p for p in pairs if (p[0]+p[1])%4!=0]; test=[p for p in pairs if p not in train]
    def tensors(rows):
        return torch.tensor(rows),torch.tensor([[0,s,c,1] for c,s in rows])
    return tensors(train),tensors(test)


In [ ]:
def checks():
    q=torch.zeros(1,2); keys=torch.tensor([[[1.,0.],[0.,2.]]])
    ctx,w=context_attention(q,keys); assert torch.allclose(ctx,torch.tensor([[.5,1.]]))
    # IDs: BOS0, EOS1, A2, B3, C4. Demonstrates greedy search error.
    def step(prefix):
        p=[1e-12]*5
        if len(prefix)==1: p[2]=.6; p[3]=.4
        elif prefix[-1]==2: p[1]=.4; p[4]=.6
        elif prefix[-1]==3: p[1]=.95; p[4]=.05
        else: p[1]=1.
        return [math.log(v) for v in p]
    assert beam_search(step,0,1,1,3)[0]==[0,2,4,1]
    assert beam_search(step,0,1,2,3)[0]==[0,3,1]


In [ ]:
def experiment():
    torch.manual_seed(17); (src,tgt),(test,truth)=data(); model=Translator()
    opt=torch.optim.Adam(model.parameters(),lr=.01)
    initial=sequence_loss(model.teacher_forced(src,tgt),tgt).item()
    for _ in range(250):
        opt.zero_grad(); loss=sequence_loss(model.teacher_forced(src,tgt),tgt); loss.backward(); opt.step()
    model.eval(); rows=[]
    for sentence,gold in zip(test,truth):
        keys,h0=model.encode(sentence[None])
        def step(prefix):
            h=h0
            for token in prefix:
                logits,h=model.decode(torch.tensor([token]),h,keys)
            return F.log_softmax(logits[0],-1).detach().tolist()
        greedy,_=beam_search(step,0,1,1,4); beam,score=beam_search(step,0,1,3,4)
        rows.append({'source':sentence.tolist(),'gold':gold.tolist(),'greedy':greedy,'beam':beam,'beam_logp':score})
    return {'initial_train_loss':initial,'final_train_loss':loss.item(),'held_out_combinations':rows,'greedy_exact_match':sum(r['greedy']==r['gold'] for r in rows)/len(rows),'beam_exact_match':sum(r['beam']==r['gold'] for r in rows)/len(rows)}


## Run checks and the supplied experiment

Complete the exercise functions first. A `NotImplementedError` here is expected in an untouched starter. The result dictionary covers the supplied baseline; the handout also requires comparisons and analysis that you must add below.


In [ ]:
import time

checks()
_started = time.perf_counter()
results = experiment()
baseline_runtime_seconds = time.perf_counter() - _started
results


## Your required comparisons

Add the additional experiments, checks, plots/tables, measured runtimes, and clearly labeled results required by the handout. Add their JSON-compatible summaries to `additional_results`.


In [ ]:
# TODO: implement and run the remaining required comparisons here.
additional_results = {}


## Brief analysis

1. Why can teacher-forced loss improve without equivalent free-running performance?

2. Is a beam gain evidence about search or about the learned distribution, and what does the controlled counterexample isolate?

3. What would be needed to extend the invented reordering task to real machine translation?

Write approximately 2–4 sentences for each question in the next cell, tied to your measured results.

1. _Your response:_

2. _Your response:_

3. _Your response:_


## Save and submit

From Colab: **Share → General access → Anyone with the link → Viewer**. Click **Copy link**. Paste the link into the Canvas assignment submission.
